<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/hybrid-rag-search-pipeline/blob/main/hybrid_search_rag_from_pdf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Install Required Libraries and Packages and Import It

In [ ]:
!pip install pymupdf langchain langchain_text_splitters qdrant-client sentence-transformers langchain-groq langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.7 MB/s eta 0:00:00


In [ ]:
import os
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer
import uuid
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

###Chunking and Splitting

####Flat chunking (Chunking using Chunk size and Overlap Size) and Get Contents

In [ ]:
def get_files_content(pdf_path):
  doc = fitz.open(pdf_path)
  text = ""

  for page in doc:
    text += page.get_text()+"\n"

  return text


In [ ]:
def chunk_text(text, chunk_size=1000, overlap_size= 200):
  chunks = []

  for i in range(0, len(text), chunk_size - overlap_size):
    chunk = text[i:i+ chunk_size]
    chunks.append(chunk)

  return chunks


In [ ]:
def recursive_text_splitter(text, chunk_size=1000, overlap_size= 200):
   splitter = RecursiveCharacterTextSplitter(
       chunk_size = chunk_size,
       chunk_overlap = overlap_size,
       separators = [
           "\n",
           "\n\n",
           ". ",
           " ",
           ""
       ]
   )

   return splitter.split_text(text)

#### Metadata Chunking

In [ ]:
def extract_pages(pdf_path):
  doc = fitz.open(pdf_path)
  pages = []

  for page_num, page in enumerate(doc, start=1):
    pages.append({
        "page": page_num,
        "text": page.get_text()
    })

  return pages

In [ ]:
def recersive_chunk_pages(pages, chunk_size = 1000, overlap_size = 200):
  splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = overlap_size,
      separators = [
           "\n",
           "\n\n",
           ". ",
           " ",
           ""
       ]
  )

  chunks = []
  id = 1

  for page in pages:
     page_chunks = splitter.split_text(page["text"])

     for chunk in page_chunks:
        chunks.append({
            "chunk_id": id,
            "text": chunk,
            "start_page": page["page"]
        })
        id += 1

  return chunks

####Overlap + Metadata chunking

In [ ]:
import fitz

def extract_pdf_with_page_map(pdf_path):
    doc = fitz.open(pdf_path)

    full_text = ""
    page_map = []

    current_pos = 0

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()

        start = current_pos
        full_text += text + "\n"
        end = len(full_text)

        page_map.append({
            "page": page_num,
            "start": start,
            "end": end
        })

        current_pos = end

    return full_text, page_map

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(text), step):
        chunk_text = text[i:i + chunk_size]

        chunks.append({
            "text": chunk_text,
            "start": i,
            "end": i + len(chunk_text)
        })

    return chunks

In [ ]:
def add_page_numbers(chunks, page_map):
    final_chunks = []

    for chunk in chunks:
        start_page = None
        end_page = None

        for page in page_map:
            # check overlap
            if chunk["start"] <= page["end"] and chunk["end"] >= page["start"]:

                if start_page is None:
                    start_page = page["page"]

                end_page = page["page"]

        final_chunks.append({
            "text": chunk["text"],
            "start_page": start_page,
            "end_page": end_page
        })

    return final_chunks

In [ ]:
full_text, page_map = extract_pdf_with_page_map('/content/Marcus-Aurelius-Meditations.pdf')
chunks = chunk_text(full_text, 1000, 200)
final_output = add_page_numbers(chunks, page_map)

### Vector DB Setup (Qdrant)

In [ ]:
def create_collection(client, collection_name, model, final_output, overwrite=False):

    if client.collection_exists(collection_name):
        if not overwrite:
            print("Collection exists. Skipping...")
            return
        else:
            client.delete_collection(collection_name)

    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )

    points = [
        PointStruct(
            id=str(uuid.uuid4()),
            vector=model.encode(c["text"], normalize_embeddings=True).tolist(),
            payload=c
        )
        for c in final_output
    ]

    client.upsert(collection_name=collection_name, points=points)

In [ ]:
def initiate_db(model_name = "BAAI/bge-small-en-v1.5", path="/content/qdrant_db", collection_name = "docs"):
  client = QdrantClient(path=path)
  model = SentenceTransformer(model_name)
  collection_name = collection_name
  create_collection(client, collection_name, model, final_output)
  return client, model

#### Retrival from Vector DB

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.payload["text"] for doc in docs)

In [ ]:
def retrive_relevant_chunks(query, model, client, collection_name):
  query_vector = model.encode(
    query,
    normalize_embeddings=True
  ).tolist()

  results = client.query_points(
      collection_name=collection_name,
      query=query_vector,
      limit=5
  ).points

  return format_docs(results), results;

###LLM Initialize

In [ ]:
GROQ_API_KEY = ''

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.6,
    api_key = GROQ_API_KEY
)

In [ ]:
def build_prompt(context, question):
    return f"""
You are a helpful assistant who explains philosophy in a very simple, clear, and human way.

Your job:
- You will receive text from an old philosophy book (Context) and a user Question.
- Rewrite and explain the idea in VERY simple language so that anyone (even a beginner) can understand it.
- Break down complex philosophical ideas into everyday examples, stories, or analogies.
- Keep the tone friendly, modern, and easy to follow.
- Avoid jargon or complicated academic language.

If context is provided:
- Use ONLY the context to answer.
- Explain the meaning in a simple, understandable way.
- Make philosophy feel practical and relatable to real life.

If context is EMPTY or NOT PROVIDED:
- Do NOT try to answer factually.
- Respond with a creative, slightly funny philosophical-style line like:
  "That philosophy has not been born yet."
  or
  "The ancient thinkers are still thinking… please try again later."
  or similar playful philosophical humor.

Context:
{context}

Question:
{question}

Answer in simple language:
"""

In [ ]:
def ask_llm(prompt_text):
    response = llm.invoke(prompt_text)
    return response.content

### RAG Pipeline

In [ ]:
model_name = "BAAI/bge-small-en-v1.5"
path="/content/qdrant_db"
collection_name = "docs"
client, model = initiate_db(model_name, path, collection_name)

In [ ]:
def rag(query):
  context, raw_result = retrive_relevant_chunks(query,
                                 model = model,
                                 collection_name = collection_name,
                                 client = client
                                 )

  prompt_text = build_prompt(context, query)
  answer = ask_llm(prompt_text)
  return answer

In [ ]:
response = rag("Why he telling that we should think about our death everyday ?")
print(response)